# Study 815 — Variance-Ratio Reversal 📏🔁

**Do the mean-reverting names (variance ratio below 1) pay you to fade them?**

Lo & MacKinlay (1988) built the **variance ratio** `VR(q) = Var(q-day return) / (q ×
Var(1-day return))` to test whether prices follow a random walk. Under the null `VR = 1`;
`VR < 1` means the return series **mean-reverts** (negative autocorrelation), `VR > 1`
means it **trends**. The cross-sectional question: rank a universe by trailing `VR(q=5)`
and buy the **low-VR (mean-reverting)** names / sell the **high-VR (trending)** ones —
does the reversal side pay? We take the self-contained daily version on a liquid US
cross-section (2010-01-04 → 2026-06-30, 50 names).

*Numbers below are the frozen headline (`docs/results.md`); the live cells run the fast
synthetic control. Survivorship: current-membership mega-caps — magnitudes are an upper
bound.*


## 1. The idea in one picture

A random walk has no memory: today's move tells you nothing about tomorrow's, and its variance ratio sits at **1**. If a name's `VR(5)` is **below 1** its recent moves have been *reversing* (up-day tends to be followed by down-day); if **above 1** they have been *trending*. The reversal trade says: buy the mean-reverters (low VR), sell the trenders (high VR) — and collect the difference.

In [1]:
import numpy as np, pandas as pd
R = dict(spread_bps=-2.69, t_nw=-2.44, lo_bps=6.1, hi_bps=8.8, gross_sharpe=-0.61,
         vr_median=0.991, vr_pct_below=52)
print('cross-section VR(5) median = %.3f  (%d%% of names below 1, i.e. mean-reverting)'
      % (R['vr_median'], R['vr_pct_below']))
print('long low-VR / short high-VR spread: %+.2f bps/day (NW t = %+.2f)'
      % (R['spread_bps'], R['t_nw']))
print('  low-VR book %+.2f bps vs high-VR book %+.2f bps'
      % (R['lo_bps'], R['hi_bps']))
print('  gross spread Sharpe (before cost): %.2f' % R['gross_sharpe'])

cross-section VR(5) median = 0.991  (52% of names below 1, i.e. mean-reverting)
long low-VR / short high-VR spread: -2.69 bps/day (NW t = -2.44)
  low-VR book +6.10 bps vs high-VR book +8.80 bps
  gross spread Sharpe (before cost): -0.61


## 2. Is the sort just lucky? A live synthetic control

We plant the effect in a seeded toy world (`edge>0`: each name gets a fixed MA(1) autocorrelation, and the mean-reverting names are *paid* a premium) and check the detector recovers it — and that it stays *silent* on the null (`edge=0`, VR varies but is unpriced). No network.

In [2]:
import os, sys
sys.path.insert(0, os.path.abspath('..'))
sys.path.insert(0, os.path.abspath(os.path.join('..','..','..')))
from variance_ratio import data, strategy as st
null = st.synthetic_detect(data.synthetic_panel(edge=0.0, seed=815, n_assets=40, n_days=1200))
planted = st.synthetic_detect(data.synthetic_panel(edge=0.0006, seed=815, n_assets=40, n_days=1600))
print('null world   : spread NW t = %+.2f  (should be ~0)' % null['t_nw'])
print('planted world: spread NW t = %+.2f  (should light up)' % planted['t_nw'])

null world   : spread NW t = +0.32  (should be ~0)
planted world: spread NW t = +9.75  (should light up)


## 3. The honest verdict — the reversal does *not* pay here (and the sign flips)

On this liquid mega-cap tape the long-low-VR / short-high-VR spread is **-2.69 bps/day** with NW *t* = **-2.44** — significant, but with the **opposite sign** to the reversal story: here the **high-VR (trending)** names actually *out-earned* the mean-reverters (low-VR book +6.10 bps vs high-VR +8.80 bps). And it is fragile — the recent era is insignificant (*t* = -1.36) and a shorter (63-day) or longer (252-day) VR window shows essentially nothing (*t* = -0.66 / -0.53). The seeded synthetic control recovers a *planted* low-VR premium cleanly (*t* = +9.75), so the machinery works — there is simply no mean-reversion premium to harvest on 50 mega-caps; the residual sign is a mega-cap-momentum artefact. **Signal: None** (the claimed reversal edge is absent), **Tradability: Mirage**.